# optimizer-state-tensor-buffers — ex3: Adam-style three-state init: m, v lists + scalar step counter

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `optimizer-state-tensor-buffers`. Running the final beacon cell reports progress against the `Optimizer: Per-param state buffers` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Per-param state buffers` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`optimizer-state-tensor-buffers`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "optimizer-state-tensor-buffers"
DD_SUBTOPIC = "Optimizer: Per-param state buffers"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## per-param state buffers — quick refresher

Per-parameter state is a list of `zeros_like(p)` tensors, one per param. Different optimizers need different buffer counts:
- SGD-momentum: ONE buffer (`b`).
- RMSprop: TWO buffers (`b`, `v`).
- Adam: THREE buffers — first moment `m`, second moment `v`, and a step counter `t_step` (a 0-d scalar tensor per param OR one global int — Adam uses the global form).

**This drill (ex3) vs prior.** ex1 allocated ONE buffer. ex2 allocated TWO. ex3 allocates Adam's full state: TWO per-param tensor buffers (`m`, `v`) PLUS a global Python-int step counter `self.t = 0`. The drill also tests the **alias-bug guard**: `self.m` and `self.v` must be SEPARATE lists of SEPARATE tensors. A common mistake is `self.v = self.m` (alias) — mutating one mutates the other.

### Exercise 3 — Adam-style three-state init: m, v lists + scalar step counter

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply per-param `zeros_like` allocation to BOTH Adam moment buffers `m` and `v` AND initialize a separate scalar step counter `self.t = 0`, with no aliasing between `m` and `v`.
> Keywords: adam-init, step-counter, alias-guard, three-buffers
> ```

**KCs targeted:** `state-buffer-multiple-buffers-per-optimizer`, `state-buffer-no-aliasing`

Implement `Ex3AdamInit.__init__(self, params)`. Adam's init-time skeleton:

1. Materialize `self.params = list(params)`.
2. Allocate the first-moment list: `self.m = [t.zeros_like(p) for p in self.params]`.
3. Allocate the second-moment list: `self.v = [t.zeros_like(p) for p in self.params]`.
4. Initialize the SCALAR step counter `self.t = 0` (Python int — Adam increments it once per `step()` call, used in the bias-correction term).

**Two failure modes the test catches.**
- **Alias bug.** If you write `self.v = self.m`, both lists share the same underlying tensors — the test asserts `id(self.m[i]) != id(self.v[i])` AND mutates `m[0]` to verify `v[0]` is unaffected.
- **Wrong step counter type.** Adam's step counter is a Python int (or sometimes a 0-d tensor), but NOT a list and NOT None. The test asserts `isinstance(self.t, int) and self.t == 0`.

**No step() body needed.** This drill isolates the init step — the same pattern as ex1 and ex2 in this folder.

In [ ]:
class Ex3AdamInit:
    def __init__(self, params):
        self.params = list(params)
        self.m = [t.zeros_like(p) for p in self.params]
        self.v = [t.zeros_like(p) for p in self.params]
        self.t = 0


<details><summary>Solution</summary>

```python
class Ex3AdamInit:
    def __init__(self, params):
        self.params = list(params)
        self.m = [t.zeros_like(p) for p in self.params]
        self.v = [t.zeros_like(p) for p in self.params]
        self.t = 0
```

**Why TWO list comprehensions.** `[t.zeros_like(p) for p in self.params]` creates fresh tensors each call. If you wrote `self.v = self.m` you'd alias the lists; if you wrote `self.v = list(self.m)` you'd alias the underlying tensors (the list is new but the references inside it are shared). Two separate comprehensions are the simplest correct allocation.

**Why an int step counter, not a tensor.** Adam reads `t` only to compute the bias correction `1 - beta**t`. A Python int is cheap, easy to increment with `self.t += 1`, and never has the device / dtype / requires_grad concerns that tensor counters have. (Some implementations DO use a 0-d tensor for state-dict serialization; that's a downstream concern beyond init.)

**`zeros_like(p).requires_grad`.** Default is `False`, which is exactly what we want — moment buffers are STATE, not learnable parameters. Autograd should never trace ops on them.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()